# Reflection Experiment Results

Win rates with 95% binomial confidence intervals for the play-reflect-transfer experiment on Werewolf.

In [ ]:
import json
import glob
import os
import math
import subprocess
import altair as alt
import pandas as pd

# Set working directory to project root (git repo root)
PROJECT_ROOT = subprocess.check_output(
    ["git", "rev-parse", "--show-toplevel"], text=True
).strip()
os.chdir(PROJECT_ROOT)
print("Working directory:", os.getcwd())


def get_wolf_win_rate(
    tag: str, roster_dir: str, filter_model_team: tuple = None
) -> dict:
    """Compute wolf win rate for a condition.

    filter_model_team: optional (model_substring, team_name) to only count games
    where that model plays that team. Used to split cross-model baselines.
    """
    files = glob.glob(f"logs/episode_{tag}_werewolves_*.json")
    if not files:
        return {"n": 0, "wolf_wins": 0, "wolf_rate": 0, "se": 0}

    wolf_wins = 0
    counted = 0
    for f in files:
        d = json.load(open(f))
        rewards = d["rewards"]
        roster_file = d.get("metadata", {}).get("roster_file", "")
        roster_path = os.path.join(roster_dir, roster_file)

        r_vals = [float(r[0]) if isinstance(r, list) else float(r) for r in rewards]

        if os.path.exists(roster_path):
            roster = json.load(open(roster_path))
            agent_teams = {a["name"]: a["team"] for a in roster["agents"]}
            agents = list(d["model_mapping"].keys())

            # Filter: only count games where the specified model plays the specified team
            if filter_model_team:
                model_sub, team_name = filter_model_team
                model_on_team = [
                    a
                    for a in roster["agents"]
                    if model_sub in a["agent_model"] and a["team"] == team_name
                ]
                if not model_on_team:
                    continue  # skip this game

            wolf_reward = sum(
                r_vals[i]
                for i, a in enumerate(agents)
                if agent_teams.get(a) == "Werewolves"
            )
            if wolf_reward > 0:
                wolf_wins += 1
            counted += 1
        else:
            neg = sum(1 for r in r_vals if r < 0)
            if neg != 2:
                wolf_wins += 1
            counted += 1

    p = wolf_wins / counted if counted > 0 else 0
    se = math.sqrt(p * (1 - p) / counted) if counted > 0 else 0
    return {"n": counted, "wolf_wins": wolf_wins, "wolf_rate": p, "se": se}

In [ ]:
# Compute all conditions
results = {}
base_dir = "experiments/rosters"

# Self-play baselines
results["selfplay_gpt5"] = get_wolf_win_rate(
    "reflect_selfplay_gpt5", f"{base_dir}/selfplay_gpt5/werewolves"
)
results["selfplay_qwen32b"] = get_wolf_win_rate(
    "reflect_selfplay_qwen32b", f"{base_dir}/selfplay_qwen32b/werewolves"
)

# Cross-model baseline: Qwen vs GPT-5 (split by side)
results["cross_qwen_wolf_vs_gpt5"] = get_wolf_win_rate(
    "reflect_qwen32b_vs_gpt5",
    f"{base_dir}/qwen32b_vs_gpt5/werewolves",
    filter_model_team=("Qwen", "Werewolves"),
)
results["cross_qwen_vlg_vs_gpt5"] = get_wolf_win_rate(
    "reflect_qwen32b_vs_gpt5",
    f"{base_dir}/qwen32b_vs_gpt5/werewolves",
    filter_model_team=("Qwen", "Villagers"),
)

# Cross-model baseline: Qwen vs GPT-4o (split by side)
results["cross_qwen_wolf_vs_gpt4o"] = get_wolf_win_rate(
    "reflect_qwen32b_vs_gpt4o",
    f"{base_dir}/qwen32b_vs_gpt4o/werewolves",
    filter_model_team=("Qwen", "Werewolves"),
)
results["cross_qwen_vlg_vs_gpt4o"] = get_wolf_win_rate(
    "reflect_qwen32b_vs_gpt4o",
    f"{base_dir}/qwen32b_vs_gpt4o/werewolves",
    filter_model_team=("Qwen", "Villagers"),
)

# GPT-5 reflection
results["gpt5_R_wolf"] = get_wolf_win_rate(
    "reflect_gpt5_R_wolf_vs_gpt5", f"{base_dir}/gpt5_R_wolf_vs_gpt5/werewolves"
)
results["gpt5_R_vlg"] = get_wolf_win_rate(
    "reflect_gpt5_R_vlg_vs_gpt5", f"{base_dir}/gpt5_R_vlg_vs_gpt5/werewolves"
)

# Qwen + reflection vs Qwen (self-improvement)
results["qwen_R_wolf_vs_qwen"] = get_wolf_win_rate(
    "reflect_qwen32b_R_wolf_vs_qwen32b",
    f"{base_dir}/qwen32b_R_wolf_vs_qwen32b/werewolves",
)
results["qwen_R_vlg_vs_qwen"] = get_wolf_win_rate(
    "reflect_qwen32b_R_vlg_vs_qwen32b",
    f"{base_dir}/qwen32b_R_vlg_vs_qwen32b/werewolves",
)

# Qwen + reflection vs GPT-5 (distillation - strong)
results["qwen_R_wolf_vs_gpt5"] = get_wolf_win_rate(
    "reflect_qwen32b_R_wolf_vs_gpt5", f"{base_dir}/qwen32b_R_wolf_vs_gpt5/werewolves"
)
results["qwen_R_vlg_vs_gpt5"] = get_wolf_win_rate(
    "reflect_qwen32b_R_vlg_vs_gpt5", f"{base_dir}/qwen32b_R_vlg_vs_gpt5/werewolves"
)

# Qwen + reflection vs GPT-4o (distillation - mid)
results["qwen_R_wolf_vs_gpt4o"] = get_wolf_win_rate(
    "reflect_qwen32b_R_wolf_vs_gpt4o", f"{base_dir}/qwen32b_R_wolf_vs_gpt4o/werewolves"
)
results["qwen_R_vlg_vs_gpt4o"] = get_wolf_win_rate(
    "reflect_qwen32b_R_vlg_vs_gpt4o", f"{base_dir}/qwen32b_R_vlg_vs_gpt4o/werewolves"
)

# Print summary
for k, v in results.items():
    print(f"{k:35s}  N={v['n']:3d}  wolf_rate={v['wolf_rate']:.2f}")

In [ ]:
def make_comparison_chart(rows: list[dict], title: str) -> alt.LayerChart:
    """Make a horizontal dot chart with error bars."""
    df = pd.DataFrame(rows)

    points = (
        alt.Chart(df)
        .mark_point(size=100, filled=True)
        .encode(
            x=alt.X("Win Rate:Q", scale=alt.Scale(domain=[0, 1]), title="Win Rate"),
            y=alt.Y("Condition:N", sort=None, title=""),
            color=alt.Color(
                "Type:N",
                scale=alt.Scale(
                    domain=["Baseline", "With Reflection"], range=["#888888", "#2a9d8f"]
                ),
            ),
            tooltip=["Condition", "Win Rate", "N"],
        )
    )

    error_bars = (
        alt.Chart(df)
        .mark_rule(strokeWidth=2)
        .encode(
            x="CI_lo:Q",
            x2="CI_hi:Q",
            y=alt.Y("Condition:N", sort=None),
            color=alt.Color(
                "Type:N",
                scale=alt.Scale(
                    domain=["Baseline", "With Reflection"], range=["#888888", "#2a9d8f"]
                ),
            ),
        )
    )

    ref_line = (
        alt.Chart(pd.DataFrame({"x": [0.5]}))
        .mark_rule(strokeDash=[4, 4], color="gray", opacity=0.5)
        .encode(x="x:Q")
    )

    return (error_bars + points + ref_line).properties(
        width=350, height=100, title=title
    )


def row(label, key, as_vlg=False):
    """Build a row dict. If as_vlg, report villager win rate instead of wolf."""
    r = results[key]
    p = (1 - r["wolf_rate"]) if as_vlg else r["wolf_rate"]
    se = r["se"]
    is_reflect = "_R_" in key
    return {
        "Condition": label,
        "Win Rate": p,
        "CI_lo": max(0, p - 1.96 * se),
        "CI_hi": min(1, p + 1.96 * se),
        "N": r["n"],
        "Type": "With Reflection" if is_reflect else "Baseline",
    }

In [ ]:
# Build one dataframe with all conditions
all_rows = []


def add_row(label, key, experiment, role, as_vlg=False):
    r = results[key]
    p = (1 - r["wolf_rate"]) if as_vlg else r["wolf_rate"]
    se = r["se"]
    is_reflect = "_R_" in key
    all_rows.append(
        {
            "Condition": label,
            "Win Rate": p,
            "CI_lo": max(0, p - 1.96 * se),
            "CI_hi": min(1, p + 1.96 * se),
            "N": r["n"],
            "Type": "With Reflection" if is_reflect else "Baseline",
            "Experiment": experiment,
            "Role": role,
        }
    )


# 1. GPT-5 self-improvement
add_row("Baseline", "selfplay_gpt5", "GPT-5 vs GPT-5", "As Werewolf")
add_row("+ Reflection", "gpt5_R_wolf", "GPT-5 vs GPT-5", "As Werewolf")
add_row("Baseline", "selfplay_gpt5", "GPT-5 vs GPT-5", "As Villager", as_vlg=True)
add_row("+ Reflection", "gpt5_R_vlg", "GPT-5 vs GPT-5", "As Villager", as_vlg=True)

# 2. Qwen self-improvement
add_row("Baseline", "selfplay_qwen32b", "Qwen32B vs Qwen32B", "As Werewolf")
add_row("+ Reflection", "qwen_R_wolf_vs_qwen", "Qwen32B vs Qwen32B", "As Werewolf")
add_row(
    "Baseline", "selfplay_qwen32b", "Qwen32B vs Qwen32B", "As Villager", as_vlg=True
)
add_row(
    "+ Reflection",
    "qwen_R_vlg_vs_qwen",
    "Qwen32B vs Qwen32B",
    "As Villager",
    as_vlg=True,
)

# 3. Distillation vs GPT-5 (strong opponent)
add_row("Baseline", "cross_qwen_wolf_vs_gpt5", "Qwen32B vs GPT-5", "As Werewolf")
add_row("+ Reflection", "qwen_R_wolf_vs_gpt5", "Qwen32B vs GPT-5", "As Werewolf")
add_row(
    "Baseline", "cross_qwen_vlg_vs_gpt5", "Qwen32B vs GPT-5", "As Villager", as_vlg=True
)
add_row(
    "+ Reflection", "qwen_R_vlg_vs_gpt5", "Qwen32B vs GPT-5", "As Villager", as_vlg=True
)

# 4. Distillation vs GPT-4o (mid opponent)
add_row("Baseline", "cross_qwen_wolf_vs_gpt4o", "Qwen32B vs GPT-4o", "As Werewolf")
add_row("+ Reflection", "qwen_R_wolf_vs_gpt4o", "Qwen32B vs GPT-4o", "As Werewolf")
add_row(
    "Baseline",
    "cross_qwen_vlg_vs_gpt4o",
    "Qwen32B vs GPT-4o",
    "As Villager",
    as_vlg=True,
)
add_row(
    "+ Reflection",
    "qwen_R_vlg_vs_gpt4o",
    "Qwen32B vs GPT-4o",
    "As Villager",
    as_vlg=True,
)

df = pd.DataFrame(all_rows)
df

In [ ]:
# Faceted chart: rows = Experiment, columns = Role
points = (
    alt.Chart(df)
    .mark_point(size=80, filled=True)
    .encode(
        x=alt.X("Win Rate:Q", scale=alt.Scale(domain=[0, 1]), title="Win Rate"),
        y=alt.Y("Condition:N", sort=["Baseline", "+ Reflection"], title=""),
        color=alt.Color(
            "Type:N",
            scale=alt.Scale(
                domain=["Baseline", "With Reflection"], range=["#888888", "#2a9d8f"]
            ),
        ),
        tooltip=["Condition", "Win Rate", "N", "CI_lo", "CI_hi"],
    )
)

error_bars = (
    alt.Chart(df)
    .mark_rule(strokeWidth=2)
    .encode(
        x="CI_lo:Q",
        x2="CI_hi:Q",
        y=alt.Y("Condition:N", sort=["Baseline", "+ Reflection"]),
        color=alt.Color(
            "Type:N",
            scale=alt.Scale(
                domain=["Baseline", "With Reflection"], range=["#888888", "#2a9d8f"]
            ),
        ),
    )
)

ref_line = (
    alt.Chart(pd.DataFrame({"x": [0.5]}))
    .mark_rule(strokeDash=[4, 4], color="gray", opacity=0.5)
    .encode(x="x:Q")
)

chart = (
    (error_bars + points + ref_line)
    .properties(width=300, height=80)
    .facet(
        column=alt.Column(
            "Role:N",
            title=None,
            sort=["As Werewolf", "As Villager"],
            header=alt.Header(labelFontSize=14, labelPadding=5),
        ),
        row=alt.Row(
            "Experiment:N",
            title=None,
            sort=[
                "GPT-5 vs GPT-5",
                "Qwen32B vs Qwen32B",
                "Qwen32B vs GPT-5",
                "Qwen32B vs GPT-4o",
            ],
            header=alt.Header(
                labelFontSize=13, labelPadding=5, labelAngle=0, labelAlign="left"
            ),
        ),
    )
    .resolve_scale(y="independent")
    .properties(title="Werewolf Reflection Experiment (30 games each, 95% CI)")
    .configure_title(fontSize=15, anchor="start")
)

chart